In [127]:
#!pip install sympy

In [128]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from qhdopt import QHD

In [129]:
# 初始化（用默认 3-bus 数据）
#model = SympyACOPFModel()   # 打开 proximal 选项，后面会用到

# 2bus model
Sbase_2 = 10.0
buses_2 = {
    1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
    2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
}
lines_2 = {
    1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase_2],
}
gens_2 = {
    1: [1, 0.0 / Sbase_2, 20.0 / Sbase_2, -20.0 / Sbase_2, 100.0 / Sbase_2, 0.00375, 2.0, 0.0],
}

model = SympyACOPFModel(Sbase = Sbase_2, buses=buses_2, lines=lines_2, gens=gens_2)

In [130]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 10.0
alpha = 5e-3   # 对偶步长尽量小一点
max_outer = 200
tol = 1e-4

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=50.0
        )

    option = 1
    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)

        qhd_model.simbi_setup(
            resolution=5,
            agents=1024,
            max_steps=1000,
            embedding_scheme="unary",
            post_processing_method='TNC',
            verbose=True
        )

        response = qhd_model.optimize(verbose=0)

        x_new = np.asarray(response.refined_minimizer)

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    PrintQHDACOPFResults(model, x_new)

    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=rho,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    rho_max = 5e3
    if np.linalg.norm(h_x) > 0.5 * np.linalg.norm(h_old) and rho < rho_max:
        rho *= 2
        print("Increasing rho to", rho)

    # ======================================
    # (7) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(xk)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (6) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")


===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.5143337249755859
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	1.0117	0.0111	0.2400	0.0839	1.0000	0.0000	0.0000	0.0000
2	0.9966	-0.0132	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.2400	0.0839		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.0257	-0.0225	0.0041	-0.0056


Total Ploss: 0.0031
Total Qloss: -0.0016
Total Load Supplied: 78.9619%
||h(x)|| = 0.2665098855073072
[rho-check] ||h_old||=4.967e-01, ||h_new||=2.665e-01, rho=10
Increasing rho to 20.0
Constraint check: False

--- Outer Iteration 1 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.49558520317077637
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	1.0171	0.0012	0.1726	0.0677	1.0000	0.0000	0.0000	0.0000
2	0.9997	-0.0252	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.1726	0.0677		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.0831	-0.0781	0.0261	-0.0240


Total Ploss: 0.0050
Total Qloss: 0.0021
Total Load Supplied: 55.8663%
||h(x)|| = 0.19131657476855926
[rho-check] ||h_old||=2.665e-01, ||h_new||=1.913e-01, rho=20
Increasing rho to 40.0
Constraint check: False

--- Outer Iteration 2 ---


...

Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.4878213405609131
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	1.0542	0.0069	0.2985	0.0823	1.0000	0.0000	0.0000	0.0000
2	1.0260	-0.0435	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.2985	0.0823		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.2931	-0.2976	0.0899	-0.0747


Total Ploss: -0.0045
Total Qloss: 0.0152
Total Load Supplied: 101.0000%
||h(x)|| = 0.034296244781887454
[rho-check] ||h_old||=7.690e-02, ||h_new||=3.430e-02, rho=5.12e+03
Constraint check: False

--- Outer Iteration 40 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.533484697341919


KeyboardInterrupt: 

In [ ]:
x_new

array([ 3.04517160e-01,  9.78303260e-02,  1.02258213e+00,  9.89472153e-01,
        6.80907997e-10, -5.03554169e-02,  1.04567422e+00,  9.81590810e-01,
        3.04517158e-01, -2.99999980e-01,  1.08496202e-01, -8.99877668e-02,
        1.04502122e-01,  9.80977834e-02])

In [ ]:
Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=np.ones_like(xk),
            rho=rho,
            ref_bus_id=None,
            mu_prox=1500.0
        )

In [ ]:
Lag

0.001875*P_G0**2 - 0.00109637901803517*P_G0 - 6.36475952795479e-5*P_ij0 - 0.000162183898628321*P_ij1 + 0.000163798776560142*Q_G0 - 1.45558442756766e-5*Q_ij0 - 0.000163813421821368*Q_ij1 + 7.90485971569055e-5*S_tot_sq0 + 5.17609295345167e-5*S_tot_sq1 + 2560.0*V_I0**2 - 0.416826690604086*V_I0 + 0.431322144778346*V_I1 + 0.10597037490987*V_R0 - 0.0912901852758375*V_R1 - 0.00374434642765875*V_sq0 - 0.00377744347781261*V_sq1 + 750.0*(P_G0 - 1.0)**2 + 750.0*(P_ij0 - 1.0)**2 + 750.0*(P_ij1 - 1.0)**2 + 750.0*(Q_G0 - 1.0)**2 + 750.0*(Q_ij0 - 1.0)**2 + 750.0*(Q_ij1 - 1.0)**2 + 750.0*(S_tot_sq0 - 1.0)**2 + 750.0*(S_tot_sq1 - 1.0)**2 + 750.0*(V_I0 - 1.0)**2 + 750.0*(V_I1 - 1.0)**2 + 750.0*(V_R0 - 1.0)**2 + 750.0*(V_R1 - 1.0)**2 + 750.0*(V_sq0 - 1.0)**2 + 750.0*(V_sq1 - 1.0)**2 + 2560.0*(-2.0*P_ij0 - 2.0*Q_ij0 + 1.0*S_tot_sq0 + 2.0)**2 + 2560.0*(-2.0*P_ij1 - 2.0*Q_ij1 + 1.0*S_tot_sq1 + 2.0)**2 + 2560.0*(-2.0*V_I0 - 2.0*V_R0 + 1.0*V_sq0 + 2.0)**2 + 2560.0*(-2.0*V_I1 - 2.0*V_R1 + 1.0*V_sq1 + 2.0)**2 +

In [ ]:
model.linear_jacobian

Matrix([
[1, 0,  5.09602092120209*V_I1 - 2.48747457492802*V_R0 + 1.24373728746401*V_R1,                         -5.09602092120209*V_I0 + 1.24373728746401*V_R0, -2.48747457492802*V_I0 + 1.24373728746401*V_I1 - 5.09602092120209*V_R1,                         1.24373728746401*V_I0 + 5.09602092120209*V_R0, 0, 0,        0,        0,        0,        0, 0, 0],
[0, 0,                         -5.09602092120209*V_I1 + 1.24373728746401*V_R1,  5.09602092120209*V_I0 + 1.24373728746401*V_R0 - 2.48747457492802*V_R1,                          1.24373728746401*V_I1 + 5.09602092120209*V_R1, 1.24373728746401*V_I0 - 2.48747457492802*V_I1 - 5.09602092120209*V_R0, 0, 0,        0,        0,        0,        0, 0, 0],
[0, 1, -1.24373728746401*V_I1 - 10.1716418424042*V_R0 + 5.09602092120209*V_R1,                          1.24373728746401*V_I0 + 5.09602092120209*V_R0, -10.1716418424042*V_I0 + 5.09602092120209*V_I1 + 1.24373728746401*V_R1,                         5.09602092120209*V_I0 - 1.24373728746401*V_R0, 0, 

In [ ]:
model.G_mat

array([[ 1.24373729, -1.24373729],
       [-1.24373729,  1.24373729]])

In [ ]:
model.arc_collection

[(0, 1), (1, 0)]

In [ ]:
response.refined_minimizer


array([ 0.31023912,  0.10302859,  1.02732258,  0.99219397, -0.03177369,
       -0.08191509,  1.04690541,  0.99452551,  0.31822289, -0.31474367,
        0.11452691, -0.09194135,  0.11429221,  0.11046397])

In [ ]:
qhd_model.func

0.001875*P_G0**2 + 4.43618087957876*P_G0 + 12.8543132406568*P_ij0 - 30.4085092598497*P_ij1 + 18.3916111650767*Q_G0 + 8.15739929272219*Q_ij0 - 12.7418433154455*Q_ij1 + 14.8464489884584*S_tot_sq0 + 13.1945724633757*S_tot_sq1 + 2560.0*V_I0**2 - 33.0144155135545*V_I0 - 23.7531556684765*V_I1 - 102.78292261944*V_R0 + 157.994071936517*V_R1 - 45.651816435364*V_sq0 + 16.9206229010081*V_sq1 + 25.0*(P_G0 - 0.310239124239775)**2 + 25.0*(P_ij0 - 0.31822288959646)**2 + 25.0*(P_ij1 + 0.314743667410533)**2 + 25.0*(Q_G0 - 0.103028586587635)**2 + 25.0*(Q_ij0 - 0.114526912753759)**2 + 25.0*(Q_ij1 + 0.0919413502177165)**2 + 25.0*(S_tot_sq0 - 0.114292213054419)**2 + 25.0*(S_tot_sq1 - 0.110463965998378)**2 + 25.0*(V_I0 + 0.0317736938323738)**2 + 25.0*(V_I1 + 0.0819150943752903)**2 + 25.0*(V_R0 - 1.02732257502023)**2 + 25.0*(V_R1 - 0.992193972650468)**2 + 25.0*(V_sq0 - 1.04690540542149)**2 + 25.0*(V_sq1 - 0.99452550834467)**2 + 2560.0*(-0.63644577919292*P_ij0 - 0.229053825507518*Q_ij0 + 1.0*S_tot_sq0 + 0.114

In [ ]:
print(model.P_D, model.Q_D, model.P_G, model.Q_G)
print(model.variable_list)
print(model.Var_bound_list)

[0.  0.3] [0.  0.1] (P_G0,) (Q_G0,)
[P_G0, Q_G0, V_R0, V_R1, V_I0, V_I1, V_sq0, V_sq1, P_ij0, P_ij1, Q_ij0, Q_ij1, S_tot_sq0, S_tot_sq1]
[[0.0, 2.0], [-2.0, 10.0], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [0.81, 1.2100000000000002], [0.81, 1.2100000000000002], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [0.0, 9.0], [0.0, 9.0]]
